# Capstone 1 — Customer Support Resolution Agent · Practice

**Goal:** build the support agent from the [README](README.md) end to end: one MCP server with four tools and structured errors, one policy layer written once, and two agent implementations on top of it (a Claude tool loop with a `PreToolUse`-style gate, and an OpenAI Agents SDK agent with tool guardrails and `needs_approval`). Then score both on a labeled ticket set.

- Theory and requirements: [README.md](README.md). Read the [Requirements](README.md#requirements) and [Architecture](README.md#architecture) sections first.
- Estimated time: 4–6 hours
- Estimated API cost: **$0 by default.** Everything runs offline. The opt-in live cells in section 10 cost roughly $0.05–0.15 for the three default tickets on both stacks, a planning estimate rather than a measurement: about 9,000–12,000 input and 200–350 output tokens per ticket on `claude-sonnet-5-5` ($2 / $10 per million input / output tokens), and under $0.01 in total on `gpt-6-luna` ($0.10 / $0.50). The offline token counts behind it assume about 4 characters per token, and the live cells print real usage.
- Prerequisites: [00-prerequisites](../../../00-prerequisites/README.md) (venv from `requirements.txt`), Claude chapters [02](../../../anthropic-claude/02-tool-use/README.md), [03](../../../anthropic-claude/03-agent-sdk/README.md), [04](../../../anthropic-claude/04-model-context-protocol/README.md), [09](../../../anthropic-claude/09-escalation-human-in-the-loop/README.md), [10](../../../anthropic-claude/10-multi-agent-error-handling/README.md) and OpenAI chapters [03](../../../openai-codex/03-agents-sdk-and-agents-api/README.md) and [09](../../../openai-codex/09-escalation-human-in-the-loop/README.md).

**How the offline mode works.** The MCP server, the policy layer, the tool loop, the Agents SDK runner, the approvals and the eval harness are all real code. Only the model is replaced by a script: `ScriptedClaude` (a real `anthropic` client whose HTTP transport replays canned responses) and `ScriptedModel` (the Agents SDK's own test model). The scripts behave like a reasonable model, including two deliberate mistakes (skipping identity verification, and asking to refund $640 directly) so you can watch the policy layer catch them. Because the model is scripted, **offline metrics validate the harness and the policy layer, not model quality.** Section 10 runs the same harness against real models.

| Flag | Default | What it turns on |
|---|---|---|
| `RUN_LIVE` | `False` | Section 10: the same tickets against the real Claude and OpenAI APIs |
| `RUN_AGENT_SDK` | `False` | Section 5: the Claude Agent SDK version (starts the bundled Claude Code CLI with your key) |

In [ ]:
import asyncio
import contextlib
import copy
import importlib.util
import json
import logging
import math
import os
import re
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, field
from itertools import count
from pathlib import Path
from typing import Any, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from mcp import Client
from mcp.types import CallToolResult, Tool
from pydantic import BaseModel, ConfigDict, ValidationError

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

RUN_LIVE = False        # True: run section 10 against the real APIs (small cost, see the header)
RUN_AGENT_SDK = False   # True: run the Claude Agent SDK version in section 5 (starts the Claude Code CLI)

for noisy in ("httpx", "httpx2"):            # keep per-request HTTP logs out of the output
    logging.getLogger(noisy).setLevel(logging.WARNING)

WORK = Path(tempfile.mkdtemp(prefix="capstone01_"))   # scratch files go here, never into the repo
print("anthropic", anthropic.__version__, "| Claude model:", CLAUDE_MODEL, "| OpenAI model:", OPENAI_MODEL)
print("scratch dir:", WORK)

In [ ]:
# Live cells need keys. Every other cell runs without them.
if RUN_LIVE or RUN_AGENT_SDK:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
    assert os.getenv("OPENAI_API_KEY") or not RUN_LIVE, "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
    print("Live mode: keys found.")
else:
    print("Offline mode: no API keys needed.")

## 1. The support backend as an MCP server

The agent never touches the database. It sees four MCP tools, the same four the exam scenario names: `get_customer`, `lookup_order`, `process_refund` and `escalate_to_human`. Two pairs are easy to confuse (`get_customer` vs `lookup_order`, and `process_refund` vs `escalate_to_human`), so each description states what the tool returns, what it does **not** do, its input format and its boundary with the look-alike. The server also enforces the business rules it can check on its own (return window, refundable amount, a $500 cap as defense in depth). [Theory](README.md#the-mcp-server-and-tool-contracts)

The server is an ordinary `mcp` 2.3 `MCPServer` file. We write it to the scratch directory so that the same file can be loaded **in memory** (sections 1–9) or launched over **stdio** by `MCPServerStdio` (section 8) and by the Agent SDK (section 5).

In [ ]:
SERVER_SRC = r'''
"""Larkspur Home support MCP server (demo data, in memory).

Four tools for a customer-support agent. Tool failures come back as isError results whose JSON
carries errorCategory / isRetryable / message / customerMessage, so the agent can decide between
retrying, fixing its arguments, explaining a policy to the customer, or escalating.
"""
import copy
import json
import re
from datetime import date
from typing import Literal

from mcp.server import MCPServer
from mcp.types import CallToolResult, TextContent, ToolAnnotations
from pydantic import BaseModel, ConfigDict, Field

TODAY = date(2026, 10, 3)            # fixed "today" keeps the demo reproducible
RETURN_WINDOW_DAYS = 30
SERVER_REFUND_CAP_CENTS = 50_000     # defense in depth: the host's policy gate should stop these first

SEED_CUSTOMERS = {
    "C-1001": {"customer_id": "C-1001", "name": "Jane Doe", "email": "jane@example.com", "status": "active",
               "order_ids": ["ORD-5001", "ORD-5002", "ORD-5003"]},
    "C-1002": {"customer_id": "C-1002", "name": "Omar Haddad", "email": "omar@example.com", "status": "active",
               "order_ids": ["ORD-6001", "ORD-6002"]},
    "C-1003": {"customer_id": "C-1003", "name": "Mei Chen", "email": "mei@example.com", "status": "active",
               "order_ids": ["ORD-6101", "ORD-6102"]},
    "C-1004": {"customer_id": "C-1004", "name": "Luca Rossi", "email": "luca@example.it", "status": "active",
               "order_ids": ["ORD-7001"]},
    "C-1005": {"customer_id": "C-1005", "name": "Ana Silva", "email": "ana@example.com", "status": "active",
               "order_ids": ["ORD-8001"]},
    "C-1006": {"customer_id": "C-1006", "name": "Jane Doe", "email": "jdoe@example.org", "status": "suspended",
               "order_ids": ["ORD-8101"]},
}


def _order(order_id, customer_id, item, status, delivered_on, total_cents, marketplace="US", carrier=None, eta=None):
    return {"order_id": order_id, "customer_id": customer_id, "item": item, "status": status,
            "delivered_on": delivered_on, "total_cents": total_cents, "refunded_cents": 0,
            "marketplace": marketplace, "carrier": carrier, "eta": eta}


SEED_ORDERS = {o["order_id"]: o for o in [
    _order("ORD-5001", "C-1001", "Ceramic mug set", "delivered", "2026-09-28", 4999),
    _order("ORD-5002", "C-1001", "Brass floor lamp", "delivered", "2026-09-20", 25000),
    _order("ORD-5003", "C-1001", "Linen duvet cover", "shipped", None, 12900, carrier="UPS", eta="2026-10-06"),
    _order("ORD-6001", "C-1002", "Espresso machine", "delivered", "2026-09-25", 64000),
    _order("ORD-6002", "C-1002", "Blender", "delivered", "2026-09-27", 17999),
    _order("ORD-6101", "C-1003", "Wireless headphones", "delivered", "2026-08-15", 8900),
    _order("ORD-6102", "C-1003", "Wool rug", "delivered", "2026-09-30", 7900),
    _order("ORD-7001", "C-1004", "Dining chairs (set of 4)", "delivered", "2026-09-22", 32000, marketplace="EU"),
    _order("ORD-8001", "C-1005", "Electric kettle", "delivered", "2026-09-29", 3900),
    _order("ORD-8101", "C-1006", "Bath towels", "delivered", "2026-09-26", 5900),
]}
FLAKY_GATEWAY = {"ORD-6102"}         # the first refund attempt on these orders times out

CUSTOMERS: dict = {}
ORDERS: dict = {}
REFUNDS: list = []
ESCALATIONS: list = []
_gateway_seen: set = set()


def reset() -> None:
    """Restore the seed data. The notebook calls this before every ticket."""
    CUSTOMERS.clear()
    CUSTOMERS.update(copy.deepcopy(SEED_CUSTOMERS))
    ORDERS.clear()
    ORDERS.update(copy.deepcopy(SEED_ORDERS))
    REFUNDS.clear()
    ESCALATIONS.clear()
    _gateway_seen.clear()


reset()


class ActionTaken(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tool: str = Field(description="Tool and key arguments, e.g. lookup_order(order_id=ORD-5001)")
    outcome: str = Field(description="What happened, in one line")


class HandoffSummary(BaseModel):
    """Everything a human specialist needs to act without reading the conversation."""
    model_config = ConfigDict(extra="forbid")
    escalation_reason: Literal["customer_requested_human", "policy_gap", "threshold_exceeded",
                               "cannot_make_progress"]
    customer_id: str | None = Field(description="Verified customer ID; null if identity was not verified")
    identity_verified: bool
    order_ids: list[str]
    issue_summary: str = Field(description="What the customer wants and why, in 1-3 sentences")
    root_cause: str | None = Field(description="What actually went wrong, if known; otherwise null")
    refund_amount_cents: int | None = Field(description="Money at stake in cents; null if none")
    actions_taken: list[ActionTaken]
    recommended_action: str
    urgency: Literal["low", "normal", "high"]


mcp = MCPServer("larkspur-support", instructions=(
    "Customer-support tools for Larkspur Home. Verify the customer with get_customer before anything else. "
    "Tool errors are JSON with errorCategory and isRetryable: retry only when isRetryable is true."))

ORDER_ID = re.compile(r"ORD-\d{4}")
EMAIL = re.compile(r"[^@\s]+@[^@\s]+\.[a-z]{2,}")


def ok(data: dict) -> CallToolResult:
    return CallToolResult(content=[TextContent(type="text", text=json.dumps(data))], structured_content=data)


def fail(category: str, message: str, retryable: bool, customer_message: str | None = None, **extra) -> CallToolResult:
    """A tool execution error the model can reason about (MCP isError + our own category fields)."""
    payload = {"errorCategory": category, "isRetryable": retryable, "message": message}
    if customer_message:
        payload["customerMessage"] = customer_message
    payload.update(extra)
    return CallToolResult(content=[TextContent(type="text", text=json.dumps(payload))],
                          structured_content=payload, is_error=True)


def _mask(email: str) -> str:
    user, domain = email.split("@")
    return f"{user[0]}***@{domain}"


@mcp.tool(annotations=ToolAnnotations(title="Get customer", read_only_hint=True))
def get_customer(email: str | None = None, name: str | None = None) -> CallToolResult:
    """Find and VERIFY the customer's account. Call this first in every conversation.

    Input: exactly one of `email` (preferred; exact match, e.g. jane@example.com) or `name`
    (full name, e.g. "Jane Doe").
    Returns, for an email match: {found, verified: true, customer_id, name, email, status, order_ids}.
    order_ids lists the customer's orders but NOT their status, items or amounts: use lookup_order for those.
    A name never verifies identity: it returns {verified: false, matches: [...]} with masked emails, because
    names are not unique. Ask the customer for the email on the account; never pick one of several matches.
    {"found": false} means no account matches. That is a valid answer, not an error.
    """
    if (email is None) == (name is None):
        return fail("validation", "Provide exactly one of email or name.", False)
    if email is not None:
        email = email.strip().lower()
        if not EMAIL.fullmatch(email):
            return fail("validation", f"{email!r} is not a valid email address (expected user@domain.com).", False)
        match = [c for c in CUSTOMERS.values() if c["email"] == email]
        if not match:
            return ok({"found": False, "email": email})
        return ok({"found": True, "verified": True, **match[0]})
    matches = [c for c in CUSTOMERS.values() if c["name"].lower() == name.strip().lower()]
    return ok({"found": bool(matches), "verified": False,
               "matches": [{"name": c["name"], "email_hint": _mask(c["email"])} for c in matches],
               "note": "A name does not verify identity. Ask the customer for the email on the account."})


@mcp.tool(annotations=ToolAnnotations(title="Look up order", read_only_hint=True))
def lookup_order(order_id: str) -> CallToolResult:
    """Fetch the details of ONE order by its order ID (format ORD-#### e.g. ORD-5001).

    Returns item, status (processing, shipped, delivered), delivery date, carrier and ETA, total and
    amount already refunded (both in cents). Use it for any question about an order's status, contents,
    price or refund eligibility. Only orders on the verified customer's account may be looked up, so
    call get_customer first. It does NOT search by email or name and changes nothing.
    {"found": false} means no such order (a valid answer). Errors: validation (bad ID format) and
    permission (an order from a marketplace this agent cannot access: escalate, do not retry).
    """
    if not ORDER_ID.fullmatch(order_id):
        return fail("validation", f"order_id {order_id!r} must look like ORD-1234.", False)
    order = ORDERS.get(order_id)
    if order is None:
        return ok({"found": False, "order_id": order_id})
    if order["marketplace"] != "US":
        return fail("permission", f"This service account lacks the orders:{order['marketplace'].lower()} scope.",
                    False, "I can't open orders from our EU store from here, so I'm passing this to a colleague who can.")
    return ok({"found": True, **order})


@mcp.tool(annotations=ToolAnnotations(title="Process refund", destructive_hint=True, idempotent_hint=False))
def process_refund(customer_id: str, order_id: str, amount_cents: int,
                   reason: Literal["damaged", "not_received", "wrong_item", "changed_mind", "other"]) -> CallToolResult:
    """Refund money to the original payment method for one DELIVERED order. IRREVERSIBLE.

    Use only after get_customer verified the customer and lookup_order showed that the order is theirs,
    delivered, and within the 30-day return window. amount_cents is in cents (4999 = $49.99) and must not
    exceed the total minus what was already refunded.
    Policy: up to $100 is paid at once; $100.01-$500 waits for a supervisor's approval (the system asks,
    you just call this tool); above $500 is never processed here, use escalate_to_human instead.
    Do NOT use it for price matching, compensation or goodwill: the policy does not cover those, escalate.
    Returns {refund_id, order_id, amount_cents, status: "refunded"} when the money has moved.
    Errors: business (not delivered, outside the window, amount too high): explain with customerMessage,
    do not retry. transient (payment gateway timeout, isRetryable true): retry the same call once.
    validation: fix the arguments.
    """
    if not ORDER_ID.fullmatch(order_id):
        return fail("validation", f"order_id {order_id!r} must look like ORD-1234.", False)
    if amount_cents <= 0:
        return fail("validation", "amount_cents must be a positive whole number of cents.", False)
    order = ORDERS.get(order_id)
    if order is None or order["customer_id"] != customer_id:
        return fail("validation", f"Order {order_id} does not belong to customer {customer_id}.", False)
    if order["marketplace"] != "US":
        return fail("permission", f"This service account lacks the orders:{order['marketplace'].lower()} scope.", False)
    if order["status"] != "delivered":
        return fail("business", f"{order_id} is {order['status']}; only delivered orders can be refunded.", False,
                    "Your order hasn't been delivered yet, so I can't refund it, but I can tell you where it is.")
    age = (TODAY - date.fromisoformat(order["delivered_on"])).days
    if age > RETURN_WINDOW_DAYS:
        return fail("business", f"Delivered {age} days ago; the refund window is {RETURN_WINDOW_DAYS} days.", False,
                    f"This order was delivered {age} days ago, outside our {RETURN_WINDOW_DAYS}-day refund window, "
                    "so I can't refund it.", alternatives=["manufacturer warranty claim"])
    remaining = order["total_cents"] - order["refunded_cents"]
    if amount_cents > remaining:
        return fail("business", f"Requested {amount_cents} cents but only {remaining} cents are refundable.", False,
                    "That's more than the amount left to refund on this order.")
    if amount_cents > SERVER_REFUND_CAP_CENTS:
        return fail("business", "Refunds above $500 cannot be processed by the agent.", False,
                    "A refund of this size needs a specialist, so I'm handing it over.", nextStep="escalate_to_human")
    if order_id in FLAKY_GATEWAY and order_id not in _gateway_seen:
        _gateway_seen.add(order_id)
        return fail("transient", "Payment gateway timed out after 5s. No money moved; the same call is safe to retry.", True)
    order["refunded_cents"] += amount_cents
    refund = {"refund_id": f"RF-{len(REFUNDS) + 1:04d}", "order_id": order_id, "amount_cents": amount_cents,
              "status": "refunded"}
    REFUNDS.append(refund)
    return ok(refund)


@mcp.tool(annotations=ToolAnnotations(title="Escalate to a human", destructive_hint=False))
def escalate_to_human(handoff: HandoffSummary) -> CallToolResult:
    """Hand the case to a human specialist and open a ticket. The specialist does NOT see this conversation,
    only this summary, so make it complete and self-contained.

    Use it when: the customer explicitly asks for a person (right away, without investigating first);
    the policy does not cover the request (e.g. price matching, compensation); a refund is above $500;
    or you cannot make progress (permission error, a transient error that persists after one retry).
    Do NOT use it for refunds up to $500 (process_refund handles those, with approval when needed) or when
    you only need more information from the customer (ask them instead).
    Returns ticket_id, queue and expected_response_hours.
    """
    ticket = {"ticket_id": f"ESC-{len(ESCALATIONS) + 1:04d}",
              "queue": "refunds-tier2" if handoff.escalation_reason == "threshold_exceeded" else "support-tier2",
              "expected_response_hours": 2 if handoff.urgency == "high" else 8}
    ESCALATIONS.append({**ticket, "handoff": handoff.model_dump()})
    return ok(ticket)


if __name__ == "__main__":
    mcp.run()  # stdio transport (used by MCPServerStdio and the Claude Agent SDK)
'''

SERVER_PATH = WORK / "support_server.py"
SERVER_PATH.write_text(SERVER_SRC.lstrip())

spec = importlib.util.spec_from_file_location("support_server", SERVER_PATH)
support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(support)          # the module-level `mcp` object can be connected in memory
print("server written to", SERVER_PATH, "| tools module loaded:", support.mcp.name)

In [ ]:
def _leaf(exc: BaseException) -> BaseException:
    while isinstance(exc, BaseExceptionGroup) and len(exc.exceptions) == 1:
        exc = exc.exceptions[0]
    return exc


@contextlib.asynccontextmanager
async def connect(server):
    """`async with Client(server)` (in memory for an MCPServer object), re-raising the real error."""
    try:
        async with Client(server) as session:
            yield session
    except BaseExceptionGroup as eg:
        raise _leaf(eg) from None


async with connect(support.mcp) as session:
    MCP_TOOLS: list[Tool] = (await session.list_tools()).tools

for t in MCP_TOOLS:
    hints = t.annotations
    print(f"{t.name:<18} read_only={hints.read_only_hint!s:<5} destructive={hints.destructive_hint!s:<5} "
          f"params={list(t.input_schema.get('properties', {}))}")
    print("   ", t.description.splitlines()[0])

The descriptions are the model's main tool-selection signal. A quick lint makes the boundaries visible: each description should say what the tool returns, when **not** to use it, and name its look-alike.

In [ ]:
CONFUSABLE = {"get_customer": "lookup_order", "lookup_order": "get_customer",
              "process_refund": "escalate_to_human", "escalate_to_human": "process_refund"}


def lint_description(tool: Tool) -> list[str]:
    """Return the missing ingredients of a tool description (empty list = looks complete)."""
    d = tool.description or ""
    checks = {
        "says what it returns": "Returns" in d or "returns" in d,
        "has a negative boundary (NOT / Do NOT)": "NOT" in d,
        f"names its look-alike ({CONFUSABLE[tool.name]})": CONFUSABLE[tool.name] in d,
        "gives an input format or example": any(k in d for k in ("format", "e.g.", "in cents")),
        "at least 3 sentences": d.count(". ") + d.count(".\n") >= 3,
    }
    return [name for name, passed in checks.items() if not passed]


for t in MCP_TOOLS:
    missing = lint_description(t)
    print(f"{t.name:<18} {'OK' if not missing else 'missing: ' + ', '.join(missing)}")

weak = Tool(name="get_customer", description="Gets customer info.", input_schema={"type": "object"})
print("\na one-line description ->", lint_description(weak))

> **Observe:** `get_customer` returns order **IDs** but not order details, and says so; `lookup_order` refuses to search by email. That one sentence on each side is what stops "where is my order?" from being answered with a customer-profile call. The same applies to `process_refund` vs `escalate_to_human`: the $500 boundary and "policy gaps go to a human" are written into **both** descriptions.

> **Try it:** delete the "Do NOT use it for price matching" sentence from `process_refund` in `SERVER_SRC`, rerun the two cells above, and see which check fails. In a live run, that sentence is what keeps a price-match request from being treated as a damaged-goods refund.

## 2. Structured errors from the tools

MCP itself only defines `isError` on a tool result. The **category** fields are our convention, carried inside the content: `errorCategory` (`transient`, `validation`, `business`, `permission`), `isRetryable`, a technical `message` and, for business errors, a `customerMessage` the agent can pass on. A query that matches nothing is **not** an error. [Theory](README.md#structured-error-contract)

In [ ]:
def payload_of(result: CallToolResult) -> dict:
    """The JSON payload of an MCP tool result. SDK-level argument errors arrive as plain text: wrap them."""
    if isinstance(result.structured_content, dict):
        return result.structured_content
    text = "\n".join(c.text for c in result.content if c.type == "text")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"errorCategory": "validation", "isRetryable": False, "message": text[:300]}


CASES = [
    ("success", "get_customer", {"email": "jane@example.com"}),
    ("valid empty result", "get_customer", {"email": "sam@example.net"}),
    ("name: not verified", "get_customer", {"name": "Jane Doe"}),
    ("validation", "lookup_order", {"order_id": "5001"}),
    ("permission", "lookup_order", {"order_id": "ORD-7001"}),
    ("business (window)", "process_refund", {"customer_id": "C-1003", "order_id": "ORD-6101", "amount_cents": 8900, "reason": "damaged"}),
    ("transient (1st try)", "process_refund", {"customer_id": "C-1003", "order_id": "ORD-6102", "amount_cents": 7900, "reason": "damaged"}),
    ("transient (retry)", "process_refund", {"customer_id": "C-1003", "order_id": "ORD-6102", "amount_cents": 7900, "reason": "damaged"}),
    ("SDK arg validation", "lookup_order", {}),
]
support.reset()
async with connect(support.mcp) as session:
    for label, tool, args in CASES:
        result = await session.call_tool(tool, args)
        p = payload_of(result)
        summary = (f"{p.get('errorCategory')}, retryable={p.get('isRetryable')}: {p.get('message', '')[:60]}"
                   if result.is_error else json.dumps(p)[:90])
        print(f"{label:<22} isError={result.is_error!s:<5} {summary}")

> **Observe:** the four categories lead to four different next steps: retry the same call (transient), change the arguments (validation), explain the rule to the customer (business), or hand over (permission). A generic `"Operation failed"` would hide that choice. The SDK's own argument validation (the last row) is plain text, so `payload_of` wraps it as a validation error. Note also that `"sam@example.net"` returns `found: false` as a **success**: "no such customer" is information, not a failure.

> **Try it:** add a `"rate_limited"` case to `lookup_order` in `SERVER_SRC` that returns `fail("transient", ..., True, retryAfterSeconds=2)`. Which field would you want the agent loop (not the model) to honor?

## 3. The policy layer, written once

The refund policy lives in **plain Python** that both stacks call. `decide()` is the pre-execution gate (`allow`, `ask` for a supervisor, or `deny` with a reason the model can act on). `observe()` runs after a call and records verified identity. `prepare()` rewrites the escalation arguments so the handoff's facts come from the audit log, not from the model's memory. The Claude loop calls these directly (a `PreToolUse`-style gate), the Agent SDK calls them from hooks, and the OpenAI agent calls them from a tool guardrail and a `needs_approval` callback. [Theory](README.md#refund-policy-enforced-in-code)

In [ ]:
AUTO_APPROVE_LIMIT_CENTS = 10_000    # $100: refunds up to this are paid at once
HARD_LIMIT_CENTS = 50_000            # $500: refunds above this are never processed by the agent


@dataclass
class SupportState:
    """Per-conversation facts the gate relies on. Code owns this; the model never writes to it."""
    verified_customer_id: str | None = None
    customer_order_ids: set[str] = field(default_factory=set)
    orders: dict[str, dict] = field(default_factory=dict)          # verified lookup_order results, by order ID
    approved_call_ids: set[str] = field(default_factory=set)
    asked: dict[str, "Decision"] = field(default_factory=dict)     # Agent SDK: tool_use_id -> the 'ask' decision
    approvals: dict[str, bool] = field(default_factory=dict)       # Agent SDK: tool_use_id -> supervisor's answer
    audit: list[dict] = field(default_factory=list)
    lock: asyncio.Lock = field(default_factory=asyncio.Lock, repr=False, compare=False)   # recheck + execute atomically


@dataclass(frozen=True)
class Decision:
    action: Literal["allow", "ask", "deny"]
    rule: str
    reason: str = ""


def bare(tool_name: str) -> str:
    """mcp__support__process_refund -> process_refund (Agent SDK names MCP tools mcp__<server>__<tool>)."""
    return tool_name.split("__")[-1]


def decide(tool_name: str, args: dict, state: SupportState) -> Decision:
    """The refund policy as code. Pure function: same inputs, same decision."""
    tool = bare(tool_name)
    if tool in ("lookup_order", "process_refund") and state.verified_customer_id is None:
        return Decision("deny", "R1-verify-first",
                        "Identity not verified. Call get_customer with the email on the account first.")
    if tool == "lookup_order" and args.get("order_id") not in state.customer_order_ids:
        return Decision("deny", "R2-own-orders-only",
                        "That order is not on the verified customer's account. Ask the customer to check the "
                        "order number. Do not say whether the order exists.")
    if tool == "process_refund":
        if args.get("customer_id") != state.verified_customer_id or args.get("order_id") not in state.customer_order_ids:
            return Decision("deny", "R2-own-orders-only",
                            "Refunds are allowed only on the verified customer's own orders, with their verified customer_id.")
        amount = int(args.get("amount_cents") or 0)
        if amount > HARD_LIMIT_CENTS:
            return Decision("deny", "R3-hard-limit",
                            f"Refunds above ${HARD_LIMIT_CENTS / 100:.0f} are never processed by the agent. Call "
                            f"escalate_to_human with escalation_reason='threshold_exceeded' and "
                            f"refund_amount_cents={amount}, then tell the customer a specialist will review it.")
        if amount > AUTO_APPROVE_LIMIT_CENTS:
            return Decision("ask", "R4-supervisor-approval",
                            f"Refunds between ${AUTO_APPROVE_LIMIT_CENTS / 100:.0f} and "
                            f"${HARD_LIMIT_CENTS / 100:.0f} need a supervisor's approval.")
    return Decision("allow", "default")


def observe(tool_name: str, args: dict, payload: dict, is_error: bool, state: SupportState) -> None:
    """PostToolUse-style: a verified get_customer establishes identity; a lookup_order of an own order is remembered."""
    if bare(tool_name) == "get_customer" and not is_error and payload.get("verified"):
        state.verified_customer_id = payload["customer_id"]
        state.customer_order_ids = set(payload["order_ids"])
        state.orders = {}
    if (bare(tool_name) == "lookup_order" and not is_error and payload.get("found")
            and payload.get("customer_id") == state.verified_customer_id):
        state.orders[payload["order_id"]] = payload


def summarize(tool: str, payload: dict, is_error: bool) -> str:
    """One audit line per call, in words a human can scan."""
    if "blockedBy" in payload:
        return f"blocked by {payload['blockedBy']}"
    if is_error:
        return f"{payload.get('errorCategory')} error: {payload.get('message', '')}"
    if tool == "get_customer":
        if not payload.get("found"):
            return "no matching account"
        return f"verified {payload['customer_id']}" if payload.get("verified") else f"{len(payload['matches'])} name matches, not verified"
    if tool == "lookup_order":
        if not payload.get("found"):
            return "no such order"
        return (f"{payload['item']}: {payload['status']}, total ${payload['total_cents'] / 100:.2f}, "
                f"refunded ${payload['refunded_cents'] / 100:.2f}")
    if tool == "process_refund":
        return f"refunded ${payload['amount_cents'] / 100:.2f} ({payload['refund_id']})"
    if tool == "escalate_to_human":
        return f"ticket {payload['ticket_id']} in {payload['queue']}"
    return json.dumps(payload)[:80]


def record(state: SupportState, tool_name: str, args: dict, decision: Decision, *, executed: bool,
           is_error: bool, payload: dict, approved: bool | None = None) -> None:
    tool = bare(tool_name)
    state.audit.append({"tool": tool, "args": args, "decision": decision.action, "rule": decision.rule,
                        "executed": executed, "approved": approved, "is_error": is_error, "payload": payload,
                        "verified_customer_id": state.verified_customer_id,
                        "summary": summarize(tool, payload, is_error)})


def actions_from_audit(audit: list[dict]) -> list[dict]:
    """The handoff's actions_taken, built from what really happened (including blocked attempts)."""
    def fmt(args: dict) -> str:
        return ", ".join(f"{k}={v}" for k, v in args.items() if v is not None and k != "handoff")
    return [{"tool": f"{e['tool']}({fmt(e['args'])})", "outcome": e["summary"]}
            for e in audit if e["tool"] != "escalate_to_human"]


def prepare(tool_name: str, args: dict, state: SupportState) -> dict:
    """updatedInput-style: overwrite the facts in a handoff with values the code knows to be true."""
    if bare(tool_name) != "escalate_to_human":
        return args
    draft = dict(args.get("handoff") or {})
    draft["customer_id"] = state.verified_customer_id            # never a model-typed ID
    draft["identity_verified"] = state.verified_customer_id is not None
    draft["order_ids"] = [o for o in draft.get("order_ids", []) if o in state.customer_order_ids]
    draft["actions_taken"] = actions_from_audit(state.audit)    # from the log, not from recollection
    amount = draft.get("refund_amount_cents")
    if amount is not None:                                       # checked against the verified order data
        refundable = sum(state.orders[o]["total_cents"] - state.orders[o]["refunded_cents"]
                         for o in draft["order_ids"] if o in state.orders)
        if not 0 < amount <= refundable:
            draft["refund_amount_cents"] = None
            draft["recommended_action"] = (f"{draft.get('recommended_action', '')} [Code check: the requested amount of "
                                           f"{amount} cents does not match a verified, looked-up order (refundable: "
                                           f"{refundable} cents); confirm the amount with the customer.]").strip()
    return {"handoff": draft}


def supervisor_review(tool_name: str, args: dict, state: SupportState | None) -> tuple[bool, str]:
    """Stand-in for the human approver: approves refunds that cite a concrete product problem."""
    approved = args.get("reason") in {"damaged", "wrong_item", "not_received"}
    return approved, ("approved" if approved else
                      "A supervisor declined this refund: it needs a concrete product problem. Offer store credit instead.")


def verified_state(customer_id: str, order_ids: list[str]) -> SupportState:
    return SupportState(verified_customer_id=customer_id, customer_order_ids=set(order_ids))


JANE = verified_state("C-1001", ["ORD-5001", "ORD-5002", "ORD-5003"])
DECISION_CASES = [
    ("refund before verification", "process_refund", {"customer_id": "C-1001", "order_id": "ORD-5001", "amount_cents": 4999}, SupportState()),
    ("look up someone else's order", "lookup_order", {"order_id": "ORD-8001"}, JANE),
    ("refund $49.99", "process_refund", {"customer_id": "C-1001", "order_id": "ORD-5001", "amount_cents": 4999}, JANE),
    ("refund $250", "process_refund", {"customer_id": "C-1001", "order_id": "ORD-5002", "amount_cents": 25000}, JANE),
    ("refund $640", "process_refund", {"customer_id": "C-1001", "order_id": "ORD-5002", "amount_cents": 64000}, JANE),
    ("model-typed customer_id", "process_refund", {"customer_id": "C-1002", "order_id": "ORD-5001", "amount_cents": 100}, JANE),
    ("Agent SDK tool name", "mcp__support__process_refund", {"customer_id": "C-1001", "order_id": "ORD-5001", "amount_cents": 4999}, JANE),
]
for label, tool, args, st in DECISION_CASES:
    d = decide(tool, args, st)
    print(f"{label:<30} -> {d.action:<5} {d.rule}")

> **Observe:** the gate never trusts an ID the model typed. `customer_id` in `process_refund` must equal the ID that a **verified** `get_customer` returned in this conversation, and only that customer's orders are visible. The server checks ownership too, but the server cannot know who was verified in *this* conversation; only the host can. That is why the policy has two layers.

> **Try it:** add a rule `R5` that denies refunds while the account `status` is `"suspended"` (customer `C-1006`). You will need `observe()` to remember the status. The homework asks you to do exactly this, with tests.

## 4. Claude implementation: a gated tool loop

The Messages API version is a bounded loop driven by `stop_reason`. Before each `tool_use` block runs, the loop asks `decide()`: `deny` becomes an `is_error` `tool_result` carrying the reason, `ask` goes to the supervisor first, `allow` executes through MCP. Every call lands in the audit log. The final answer is a JSON `TicketOutcome` (structured outputs via `output_config.format`, which the docs allow together with tools), so the eval can read the status without parsing prose. [Theory](README.md#claude-implementation)

In [ ]:
_ids = count(1)


def est_tokens(obj: Any) -> int:
    """Rough token estimate (about 4 characters per token). Good enough to compare designs offline."""
    return max(1, math.ceil(len(json.dumps(obj, default=str)) / 4))


def text(t: str) -> dict:
    return {"type": "text", "text": t}


def tool_call(name: str, tool_input: dict) -> dict:
    return {"type": "tool_use", "id": f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict) -> dict:
    """A Messages API response body; stop_reason follows from the blocks."""
    stop = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": list(blocks), "stop_reason": stop, "stop_sequence": None,
            "usage": {"input_tokens": 0, "output_tokens": 0}}


class ScriptedClaude:
    """Offline Messages API: a real AsyncAnthropic client whose transport replays scripted replies.

    Usage numbers are estimated from what was actually sent and returned, so token and cost
    comparisons between designs still mean something offline.
    """

    def __init__(self, *replies: dict):
        self.replies = list(replies)
        self.requests: list[dict] = []
        self.client = anthropic.AsyncAnthropic(
            api_key="offline", max_retries=0,
            http_client=anthropic.DefaultAsyncHttpxClient(transport=httpx2.MockTransport(self._handle)))

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        self.requests.append(body)
        if not self.replies:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        out = copy.deepcopy(self.replies.pop(0))
        if out == "TIMEOUT":                       # simulate a model call that never answers
            raise httpx2.ReadTimeout("scripted read timeout", request=request)
        sent = {k: body.get(k) for k in ("system", "tools", "messages", "output_config")}
        out["usage"] = {"input_tokens": est_tokens(sent), "output_tokens": est_tokens(out["content"])}
        return httpx2.Response(200, json=out)


print("offline Claude toolkit ready")

In [ ]:
class ConcernResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    concern: str
    status: Literal["resolved", "declined_by_policy", "needs_customer_info", "escalated"]
    resolution: str


class TicketOutcome(BaseModel):
    """The agent's final answer: machine-readable for the eval, plus the message for the customer."""
    model_config = ConfigDict(extra="forbid")
    status: Literal["resolved", "needs_customer_info", "escalated"]
    concerns: list[ConcernResult]
    customer_message: str


OUTCOME_SCHEMA = TicketOutcome.model_json_schema()

SYSTEM_PROMPT = """You are the Larkspur Home customer-support agent. Resolve the customer's issue in this conversation whenever the policy allows it.

How to work
1. Split the message into separate concerns. Investigate each one; make independent lookups in parallel.
2. Verify identity with get_customer (by email) before looking up orders or refunding. If a name matches several customers, or nothing matches, ask for the email on the account. Never guess.
3. Refunds: delivered orders, within 30 days of delivery, up to the amount left on the order. Up to $100 is paid at once. Up to $500 needs a supervisor: the system asks them, you just call process_refund. Above $500 goes to escalate_to_human.

When to escalate (escalate_to_human)
- The customer asks for a person: escalate right away, without investigating first.
- The policy does not cover the request (for example price matching or compensation).
- A refund above $500.
- You cannot make progress: a permission error, or a transient error that persists after one retry.
Do not escalate only because the customer is upset: acknowledge it and resolve what you can.

Tool errors are JSON. Retry once if isRetryable is true. After a validation error, fix your arguments. Explain business errors with customerMessage. Never show internal error text to the customer.

Finish with the JSON outcome: one entry per concern, the overall status, and the message to the customer."""


def strictify(schema: Any) -> Any:
    """MCP-generated schema -> Claude strict tool schema: every object closed (additionalProperties: false).

    Strict mode accepts `default`, but we drop `title` and `default` keywords to keep the schema lean.
    Keys inside `properties` / `$defs` are names, not keywords, so a property called "title" survives.
    """
    if isinstance(schema, dict):
        out = {}
        for k, v in schema.items():
            if k in ("properties", "$defs", "definitions") and isinstance(v, dict):
                out[k] = {name: strictify(sub) for name, sub in v.items()}
            elif k not in ("title", "default"):
                out[k] = strictify(v)
        if out.get("type") == "object":
            out["additionalProperties"] = False
        return out
    if isinstance(schema, list):
        return [strictify(v) for v in schema]
    return schema


def to_claude_tool(tool: Tool) -> dict:
    """An MCP tool definition is a Claude tool definition, give or take field names."""
    return {"name": tool.name, "description": tool.description or "", "input_schema": strictify(tool.input_schema),
            "strict": True}


CLAUDE_TOOLS = [to_claude_tool(t) for t in MCP_TOOLS]
print([t["name"] for t in CLAUDE_TOOLS], "| outcome fields:", list(OUTCOME_SCHEMA["properties"]))

In [ ]:
@dataclass
class TicketRun:
    ticket_id: str
    stack: str
    status: str                        # TicketOutcome.status, or "failed"
    outcome: TicketOutcome | None
    audit: list[dict]
    approvals_requested: int
    input_tokens: int
    output_tokens: int
    turns: int

    @property
    def refunds_cents(self) -> int:
        return sum(e["payload"]["amount_cents"] for e in self.audit
                   if e["tool"] == "process_refund" and e["executed"] and not e["is_error"])

    @property
    def handoff(self) -> dict | None:
        for e in self.audit:
            if e["tool"] == "escalate_to_human" and e["executed"] and not e["is_error"]:
                return e["args"]["handoff"]
        return None


async def call_backend(session, tool_name: str, args: dict, state: SupportState, decision: Decision,
                       approved: bool | None = None) -> tuple[dict, bool]:
    """Execute one allowed call through MCP: prepare -> call -> audit -> observe."""
    args = prepare(tool_name, args, state)
    result = await session.call_tool(bare(tool_name), args)
    payload, is_error = payload_of(result), bool(result.is_error)
    record(state, tool_name, args, decision, executed=True, is_error=is_error, payload=payload, approved=approved)
    observe(tool_name, args, payload, is_error, state)
    return payload, is_error


async def fallback_escalation(session, state: SupportState, why: str) -> None:
    """When the loop itself fails, code (not the model) hands the case to a human."""
    handoff = {"escalation_reason": "cannot_make_progress", "customer_id": None, "identity_verified": False,
               "order_ids": sorted(state.customer_order_ids), "issue_summary": f"Agent run ended early: {why}",
               "root_cause": None, "refund_amount_cents": None, "actions_taken": [],
               "recommended_action": "Read the audit log and contact the customer.", "urgency": "normal"}
    await call_backend(session, "escalate_to_human", {"handoff": handoff}, state, Decision("allow", "fallback"))


def tool_result(tool_use_id: str, payload: dict, is_error: bool) -> dict:
    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": json.dumps(payload), "is_error": is_error}


async def run_claude(llm, session, ticket_id: str, text_in: str, *, gated: bool = True,
                     review=supervisor_review, max_turns: int = 10) -> TicketRun:
    """One ticket, one conversation: the stop_reason loop with a PreToolUse-style policy gate."""
    support.reset()
    state, usage, approvals = SupportState(), Counter(), 0
    messages: list[dict] = [{"role": "user", "content": text_in}]
    why, turns = f"max_turns={max_turns} reached", max_turns
    for turn in range(1, max_turns + 1):
        try:
            response = await llm.messages.create(   # max_tokens leaves room for adaptive thinking (on by default)
                model=CLAUDE_MODEL, max_tokens=16000, system=SYSTEM_PROMPT, tools=CLAUDE_TOOLS, messages=messages,
                output_config={"format": {"type": "json_schema", "schema": OUTCOME_SCHEMA}})
        except anthropic.APIError as exc:            # timeout, connection, 4xx/5xx after the SDK's own retries (E4)
            why, turns = f"model call failed: {type(exc).__name__}", turn
            break
        usage["in"] += response.usage.input_tokens
        usage["out"] += response.usage.output_tokens
        messages.append({"role": "assistant", "content": response.content})   # the FULL content, not just text

        if response.stop_reason == "end_turn":
            raw = "".join(b.text for b in response.content if b.type == "text")
            try:
                outcome = TicketOutcome.model_validate_json(raw)
            except ValidationError:
                why, turns = "final answer did not match the outcome schema", turn
                break
            return TicketRun(ticket_id, "claude", outcome.status, outcome, state.audit, approvals,
                             usage["in"], usage["out"], turn)
        if response.stop_reason != "tool_use":
            why, turns = f"stop_reason={response.stop_reason}", turn   # max_tokens, refusal, ...: never run a truncated turn
            break

        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            decision = decide(block.name, block.input, state) if gated else Decision("allow", "ungated")
            approved = None
            if decision.action == "ask":
                approvals += 1
                approved, note = review(block.name, block.input, state)
                if not approved:
                    decision = Decision("deny", "human-reviewer", note)
            if decision.action == "deny":
                blocked = {"blockedBy": decision.rule, "message": decision.reason}
                record(state, block.name, block.input, decision, executed=False, is_error=True, payload=blocked)
                results.append(tool_result(block.id, blocked, True))
                continue
            payload, is_error = await call_backend(session, block.name, block.input, state, decision, approved)
            results.append(tool_result(block.id, payload, is_error))
        messages.append({"role": "user", "content": results})          # all results in ONE user message

    await fallback_escalation(session, state, why)
    return TicketRun(ticket_id, "claude", "failed", None, state.audit, approvals, usage["in"], usage["out"], turns)


def show_run(run: TicketRun) -> None:
    marks = {"allow": "  ", "ask": "? ", "deny": "x "}
    for e in run.audit:
        mark = marks[e["decision"]] if e["executed"] or e["decision"] != "allow" else "  "
        flag = " (approved by supervisor)" if e["approved"] else ""
        print(f"  {mark}{e['tool']:<18} {e['summary']}{flag}")
    print(f"  => status={run.status}, refunded=${run.refunds_cents / 100:.2f}, escalated={run.handoff is not None}, "
          f"approvals={run.approvals_requested}, turns={run.turns}, tokens in/out={run.input_tokens}/{run.output_tokens}")
    if run.outcome:
        print("  customer:", run.outcome.customer_message)

### The scripted "model" and the first ticket (offline)

A plan is a list of turns; each turn is either tool calls (several calls in one turn are parallel) or the final outcome. The same plans drive both stacks, so any difference in results comes from the stacks, not the script.

In [ ]:
def call(name: str, /, **args: Any) -> tuple:
    return ("call", name, args)


def say(status: str, message: str, *concerns: tuple[str, str, str]) -> tuple:
    outcome = TicketOutcome(status=status, customer_message=message,
                            concerns=[ConcernResult(concern=c, status=s, resolution=r) for c, s, r in concerns])
    return ("say", outcome.model_dump_json())


def to_claude_script(plan: list[list[tuple]]) -> list[dict]:
    replies = []
    for turn in plan:
        if turn[0][0] == "say":
            replies.append(reply(text(turn[0][1])))
        else:
            replies.append(reply(*[tool_call(name, args) for _, name, args in turn]))
    return replies


T01_TEXT = "Hi, the mug set from order ORD-5001 arrived chipped. Can I get a refund? My email is jane@example.com."
T01_PLAN = [
    [call("get_customer", email="jane@example.com")],
    [call("lookup_order", order_id="ORD-5001")],
    [call("process_refund", customer_id="C-1001", order_id="ORD-5001", amount_cents=4999, reason="damaged")],
    [say("resolved", "Sorry about the chipped mugs! I've refunded $49.99 to your original payment method.",
         ("Refund for chipped mug set (ORD-5001)", "resolved", "Refunded $49.99 (within the $100 auto limit)."))],
]

fake = ScriptedClaude(*to_claude_script(T01_PLAN))
async with connect(support.mcp) as session:
    run = await run_claude(fake.client, session, "T01", T01_TEXT)
show_run(run)
print("\nrequests sent:", len(fake.requests), "| last request ends with:",
      [b["type"] for b in fake.requests[-1]["messages"][-1]["content"]])

> **Observe:** four requests for three tool turns plus the answer. The loop ended because `stop_reason` was `end_turn`, not because of a turn count or a keyword in the text. Each `tool_result` went back in a single `user` message right after the assistant turn that asked for it.

### The gate catches a model mistake

Ticket T13: the customer gives an order number and an email, and the "model" calls `process_refund` straight away with a customer ID it guessed. Run it with and without the gate.

In [ ]:
T13_TEXT = "My kettle ORD-8001 leaks. Refund the $39 now please, ana@example.com"
T13_PLAN = [
    [call("process_refund", customer_id="C-1005", order_id="ORD-8001", amount_cents=3900, reason="damaged")],
    [call("get_customer", email="ana@example.com")],
    [call("lookup_order", order_id="ORD-8001")],
    [call("process_refund", customer_id="C-1005", order_id="ORD-8001", amount_cents=3900, reason="damaged")],
    [say("resolved", "I'm sorry the kettle leaks. I've refunded $39.00 to your original payment method.",
         ("Refund for leaking kettle (ORD-8001)", "resolved", "Refunded $39.00 after verifying the account."))],
]
async with connect(support.mcp) as session:
    for gated in (True, False):
        run = await run_claude(ScriptedClaude(*to_claude_script(T13_PLAN)).client, session, "T13", T13_TEXT, gated=gated)
        print(f"gated={gated}")
        show_run(run)

> **Observe:** with the gate, the first refund is blocked by `R1-verify-first` and the model's later, correct sequence succeeds. Without it, money moves **before** anyone verified who is asking (the customer ID was a guess that happened to be right), and the second attempt fails as a business error. The final message is identical in both runs, which is exactly why you can't judge policy compliance from the transcript's last message: you need the audit log.

> **Try it:** change the guessed `customer_id` in the first step to `"C-1001"` and rerun. Which layer stops it in the ungated run?

### Errors in the loop: transient retry, business explanation, permission escalation

In [ ]:
T09_TEXT = "The rug from ORD-6102 arrived with a big stain. Refund please ($79). mei@example.com"
T09_PLAN = [
    [call("get_customer", email="mei@example.com")],
    [call("lookup_order", order_id="ORD-6102")],
    [call("process_refund", customer_id="C-1003", order_id="ORD-6102", amount_cents=7900, reason="damaged")],
    [call("process_refund", customer_id="C-1003", order_id="ORD-6102", amount_cents=7900, reason="damaged")],
    [say("resolved", "So sorry about the stain. I've refunded $79.00 to your original payment method.",
         ("Refund for stained rug (ORD-6102)", "resolved", "Refunded $79.00 after one retry of a gateway timeout."))],
]
T07_TEXT = "Please refund my headphones, order ORD-6101. They stopped charging. mei@example.com"
T07_PLAN = [
    [call("get_customer", email="mei@example.com")],
    [call("lookup_order", order_id="ORD-6101")],
    [call("process_refund", customer_id="C-1003", order_id="ORD-6101", amount_cents=8900, reason="damaged")],
    [say("resolved", "I'm sorry your headphones stopped charging. They were delivered 49 days ago, outside our "
         "30-day refund window, so I can't refund them. The manufacturer's warranty should cover this.",
         ("Refund for headphones (ORD-6101)", "declined_by_policy", "Outside the 30-day window; pointed to warranty."))],
]
async with connect(support.mcp) as session:
    for tid, text_in, plan in [("T09", T09_TEXT, T09_PLAN), ("T07", T07_TEXT, T07_PLAN)]:
        print(tid)
        show_run(await run_claude(ScriptedClaude(*to_claude_script(plan)).client, session, tid, text_in))

> **Observe:** the transient error carried `isRetryable: true`, so one identical retry was the right move and it succeeded. The business error carried `isRetryable: false` and a `customerMessage`, so the right move was to explain, and the ticket still counts as **resolved**: the customer got a definitive answer in one contact. A declined refund is not a failure of the agent.

### When the loop itself fails: truncated turns, an invalid final answer, a failed model call

A response cut off by `max_tokens` can carry a `tool_use` block with incomplete input, and a `refusal` can stop a turn half-way. The loop must not execute either; here it ends the run and **code** files the escalation. The same happens when the final answer does not match `TicketOutcome`, and when the model call itself fails (a timeout, a connection error or a 5xx after the SDK's own retries): `run_claude` catches `anthropic.APIError` at the ticket boundary, and `run_openai` does the same for `openai.APIError` and the Agents SDK's `AgentsException`. That is rule E4: the loop's own failures are escalated by code. On Claude Sonnet 5.5, a `refusal` comes from a safety classifier, and the API offers server-side fallback (`fallbacks: "default"`, beta) that retries a refused request on another model. This capstone deliberately does not use it: a support ticket the model declined is one a person should see, so code escalates it. That is a design choice for this client, not a rule; if you add fallback, keep the gate in front of whichever model answers.

In [ ]:
truncated = reply(text("Let me refund that for you."),
                  tool_call("process_refund", {"customer_id": "C-1001", "order_id": "ORD-5001"}))   # no amount, no reason
truncated["stop_reason"] = "max_tokens"
FAILURES = [("truncated turn (max_tokens)", [truncated]), ("final answer not a TicketOutcome", [reply(text("All done!"))]),
            ("model call timed out", ["TIMEOUT"])]
async with connect(support.mcp) as session:
    for label, script in FAILURES:
        run = await run_claude(ScriptedClaude(*script).client, session, "TX", T01_TEXT)
        executed = [e["tool"] for e in run.audit if e["tool"] != "escalate_to_human"]
        print(f"{label:<34} status={run.status}, tools executed={executed}, "
              f"fallback escalation={run.handoff['escalation_reason'] if run.handoff else None}")

> **Observe:** no failure executed a tool, and each produced a `cannot_make_progress` ticket filed by `fallback_escalation()`, not by the model. In production you might first retry a `max_tokens` turn once with a larger budget; what you must never do is run the truncated `tool_use`.

## 5. The same gate as Agent SDK hooks (offline)

With the Claude Agent SDK, the loop above is built in. The policy plugs in as hooks: `PreToolUse` returns `deny` (with a reason Claude reads), `ask` (routed to your `can_use_tool` callback, here the supervisor) or `allow` with `updatedInput` (the finalized handoff); `PostToolUse` records verified identity, and `PostToolUseFailure` puts failed calls in the audit log. The MCP server is the **same file**, launched over stdio, and its tools appear as `mcp__support__<tool>`. We test the hooks by calling them directly, which does not start the SDK. [Theory](README.md#claude-implementation)

In [ ]:
from claude_agent_sdk import (ClaudeAgentOptions, HookMatcher, PermissionResultAllow, PermissionResultDeny,
                              ToolPermissionContext)

SDK_SESSIONS: dict[str, SupportState] = {}
PENDING_ASKS: dict[str, SupportState] = {}      # tool_use_id -> conversation state, while a supervisor decides


def state_for(input_data: dict) -> SupportState:
    return SDK_SESSIONS.setdefault(input_data["session_id"], SupportState())


async def policy_gate_hook(input_data, tool_use_id, context):
    """PreToolUse: the same decide()/prepare() as the Messages API loop."""
    state = state_for(input_data)
    name, args = input_data["tool_name"], input_data["tool_input"]
    decision = decide(name, args, state)
    if decision.action != "allow":
        if decision.action == "deny":
            record(state, name, args, decision, executed=False, is_error=True,
                   payload={"blockedBy": decision.rule, "message": decision.reason})
        else:                                       # 'ask': remember it so the supervisor's answer can be audited
            state.asked[tool_use_id] = decision
            PENDING_ASKS[tool_use_id] = state
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": decision.action,
                                       "permissionDecisionReason": decision.reason}}
    updated = prepare(name, args, state)
    out = {"hookEventName": "PreToolUse", "permissionDecision": "allow"}   # explicit allow: no prompt
    if updated != args:
        out["updatedInput"] = updated
    return {"hookSpecificOutput": out}


def sdk_payload(tool_response: Any) -> dict:
    """Best-effort JSON from an MCP tool_response as the SDK reports it (string, content list or dict)."""
    if isinstance(tool_response, dict) and "content" in tool_response:
        tool_response = tool_response["content"]
    if isinstance(tool_response, list):
        tool_response = "".join(b.get("text", "") for b in tool_response if isinstance(b, dict))
    if isinstance(tool_response, str):
        try:
            return json.loads(tool_response)
        except json.JSONDecodeError:
            return {"raw": tool_response}
    return tool_response if isinstance(tool_response, dict) else {"raw": str(tool_response)}


def executed_decision(state: SupportState, tool_use_id: str) -> tuple[Decision, bool | None]:
    """The gate's decision for a call that ran: 'ask' plus the supervisor's answer, or a plain 'allow'."""
    if tool_use_id in state.asked:
        return state.asked.pop(tool_use_id), state.approvals.pop(tool_use_id, None)
    return Decision("allow", "default"), None


async def observe_hook(input_data, tool_use_id, context):
    """PostToolUse: record the call and, after a verified get_customer, the identity. 'Not found' is a success."""
    state = state_for(input_data)
    payload = sdk_payload(input_data["tool_response"])
    is_error = "errorCategory" in payload          # defensive: our error JSON, however the SDK reports it
    observe(input_data["tool_name"], input_data["tool_input"], payload, is_error, state)
    decision, approved = executed_decision(state, tool_use_id)
    record(state, input_data["tool_name"], input_data["tool_input"], decision,
           executed=True, is_error=is_error, payload=payload, approved=approved)
    return {}


async def failure_hook(input_data, tool_use_id, context):
    """PostToolUseFailure: failed calls belong in the audit log too (FR-8)."""
    state = state_for(input_data)
    payload = sdk_payload(input_data.get("error", ""))
    if "errorCategory" not in payload:
        payload = {"errorCategory": "unknown", "isRetryable": False, "message": str(input_data.get("error", ""))[:300]}
    decision, approved = executed_decision(state, tool_use_id)
    record(state, input_data["tool_name"], input_data["tool_input"], decision,
           executed=True, is_error=True, payload=payload, approved=approved)
    return {}


async def supervisor_can_use_tool(tool_name, tool_input, context: ToolPermissionContext):
    """Receives the calls the hook answered with 'ask'; the decision is audited under the same tool_use_id (FR-8)."""
    approved, note = supervisor_review(tool_name, tool_input, None)
    state = PENDING_ASKS.pop(context.tool_use_id, None)
    if state is not None:
        if approved:
            state.approvals[context.tool_use_id] = True          # PostToolUse records it with the executed call
        else:
            state.asked.pop(context.tool_use_id, None)
            record(state, tool_name, tool_input, Decision("deny", "human-reviewer", note), executed=False,
                   is_error=True, payload={"blockedBy": "human-reviewer", "message": note})
    return PermissionResultAllow() if approved else PermissionResultDeny(message=note)


AGENT_SDK_OPTIONS = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=SYSTEM_PROMPT,
    tools=[],                                   # no built-in tools (Bash, Read, ...) for a support agent
    mcp_servers={"support": {"type": "stdio", "command": sys.executable, "args": [str(SERVER_PATH)]}},
    allowed_tools=["mcp__support__get_customer", "mcp__support__lookup_order", "mcp__support__escalate_to_human"],
    hooks={"PreToolUse": [HookMatcher(matcher="^mcp__support__", hooks=[policy_gate_hook])],
           "PostToolUse": [HookMatcher(matcher="^mcp__support__", hooks=[observe_hook])],
           "PostToolUseFailure": [HookMatcher(matcher="^mcp__support__", hooks=[failure_hook])]},
    can_use_tool=supervisor_can_use_tool,
    max_turns=10,
)
print("options built (nothing started). hooks on:", list(AGENT_SDK_OPTIONS.hooks))

In [ ]:
# Offline unit test: feed the hooks the inputs the SDK would send.
SDK_SESSIONS.clear()


def hook_input(tool: str, args: dict, **extra) -> dict:
    return {"session_id": "sess-1", "transcript_path": "", "cwd": str(WORK), "hook_event_name": "PreToolUse",
            "tool_name": f"mcp__support__{tool}", "tool_input": args, "tool_use_id": "toolu_x", **extra}


refund = {"customer_id": "C-1001", "order_id": "ORD-5002", "amount_cents": 25000, "reason": "damaged"}
print("refund before verification:", (await policy_gate_hook(hook_input("process_refund", refund), "toolu_x", None))
      ["hookSpecificOutput"]["permissionDecision"])

verified = {"found": True, "verified": True, **support.SEED_CUSTOMERS["C-1001"]}
await observe_hook(hook_input("get_customer", {"email": "jane@example.com"},
                              tool_response=[{"type": "text", "text": json.dumps(verified)}]), "toolu_y", None)
print("after PostToolUse on get_customer: verified =", SDK_SESSIONS["sess-1"].verified_customer_id)

for amount in (4999, 64000):
    out = await policy_gate_hook(hook_input("process_refund", {**refund, "amount_cents": amount}), "toolu_x", None)
    print(f"refund {amount:>6} cents ->", out["hookSpecificOutput"]["permissionDecision"])

# The $250 refund: PreToolUse says 'ask', can_use_tool asks the supervisor, PostToolUse audits the approval.
out = await policy_gate_hook(hook_input("process_refund", refund, tool_use_id="toolu_r"), "toolu_r", None)
print(f"refund  25000 cents ->", out["hookSpecificOutput"]["permissionDecision"])
answer = await supervisor_can_use_tool("mcp__support__process_refund", refund, ToolPermissionContext(tool_use_id="toolu_r"))
print("supervisor callback:", type(answer).__name__)
paid = {"refund_id": "RF-0001", "order_id": "ORD-5002", "amount_cents": 25000, "status": "refunded"}
await observe_hook(hook_input("process_refund", refund, hook_event_name="PostToolUse", tool_use_id="toolu_r",
                              tool_response=[{"type": "text", "text": json.dumps(paid)}]), "toolu_r", None)
last = SDK_SESSIONS["sess-1"].audit[-1]
print("audit for the paid refund:", last["decision"], last["rule"], "approved =", last["approved"])

esc = await policy_gate_hook(hook_input("escalate_to_human", {"handoff": {"customer_id": "C-9999", "order_ids": ["ORD-5002"]}}),
                             "toolu_z", None)
print("escalation rewritten via updatedInput: customer_id =", esc["hookSpecificOutput"]["updatedInput"]["handoff"]["customer_id"])
await failure_hook(hook_input("lookup_order", {"order_id": "ORD-7001"}, hook_event_name="PostToolUseFailure",
                              error=json.dumps({"errorCategory": "permission", "isRetryable": False, "message": "scope"})),
                   "toolu_f", None)
print("audit after a failed call:", [(e["tool"], e["summary"]) for e in SDK_SESSIONS["sess-1"].audit][-1])

In [ ]:
# (live, opt-in) The real Agent SDK run. Starts the bundled Claude Code CLI and the MCP server over stdio.
if RUN_AGENT_SDK:
    from claude_agent_sdk import ClaudeSDKClient, ResultMessage

    SDK_SESSIONS.clear()
    async with ClaudeSDKClient(options=AGENT_SDK_OPTIONS) as sdk:
        await sdk.query("The brass floor lamp (ORD-5002) arrived with a dented base. I'd like a full refund "
                        "of $250. Email jane@example.com")
        async for msg in sdk.receive_response():
            if isinstance(msg, ResultMessage):
                print("result:", msg.subtype, "| cost:", msg.total_cost_usd, "| denials:", msg.permission_denials)
                print(msg.result)
    for st in SDK_SESSIONS.values():
        for e in st.audit:
            print(" ", e["tool"], "->", e["summary"])
else:
    print("Skipped: set RUN_AGENT_SDK = True to run the Agent SDK version (uses your Anthropic key).")

> **Observe:** the hook returns an **explicit** `allow` for in-policy calls. That matters for `process_refund`, which is deliberately not in `allowed_tools`: in-policy refunds are approved by the hook, the $100–$500 ones are routed to `can_use_tool` by `ask`, and anything above $500 is denied before a person is bothered. The supervisor's answer is part of the audit trail (FR-8): the hook files each `ask` under its `tool_use_id`, `can_use_tool` receives the same ID in `ToolPermissionContext.tool_use_id`, and `PostToolUse` records the paid refund as `ask` / `R4-supervisor-approval` with `approved = True`, exactly like the Messages API loop (a rejection is recorded as `human-reviewer`). Without that link the eval would flag every approved refund as unapproved. For approvals that take hours, return `defer` instead of `ask` and resume the session later ([Chapter 9, section 9.5](../../../anthropic-claude/09-escalation-human-in-the-loop/README.md#95-human-approval-checkpoints-on-each-claude-surface-expanded-beyond-the-guide)).

> **What the docs pin down, and what they don't:** the hooks reference says an MCP tool that returns an error result fires `PostToolUseFailure`, not `PostToolUse`, and passes an `error` string that is generally the text Claude receives but has no stable format. A successful call fires `PostToolUse` with a `tool_response` whose exact shape "depends on the tool". That is why `failure_hook` exists and why `sdk_payload()` accepts a string, a content list or a dict. **Try it:** when you run the live cell, print one raw `tool_response` in `observe_hook` and one raw `error` in `failure_hook`, then simplify the function.

## 6. Escalation with a structured handoff

The specialist who picks up an escalation cannot see the conversation. The handoff therefore *is* the input schema of `escalate_to_human`, and code overwrites the facts the model might get wrong: `customer_id` and `identity_verified` come from the verified state, `actions_taken` from the audit log, and unverified order IDs are dropped. [Theory](README.md#structured-handoff)

In [ ]:
def handoff_draft(reason: str, summary: str, *, orders: list[str], amount: int | None, root_cause: str | None,
                  action: str, urgency: str = "normal", customer_id: str | None = None,
                  actions: list[dict] | None = None) -> dict:
    """What the model writes. Some fields will be replaced by prepare()."""
    return {"handoff": {"escalation_reason": reason, "customer_id": customer_id, "identity_verified": True,
                        "order_ids": orders, "issue_summary": summary, "root_cause": root_cause,
                        "refund_amount_cents": amount, "actions_taken": actions or [],
                        "recommended_action": action, "urgency": urgency}}


T04_TEXT = "My espresso machine ORD-6001 stopped working after a week. I want my $640 back. omar@example.com"
T04_PLAN = [
    [call("get_customer", email="omar@example.com")],
    [call("lookup_order", order_id="ORD-6001")],
    [call("process_refund", customer_id="C-1002", order_id="ORD-6001", amount_cents=64000, reason="damaged")],
    [call("escalate_to_human", **handoff_draft(
        "threshold_exceeded", "Customer wants a full $640 refund for an espresso machine that failed after a week.",
        orders=["ORD-6001"], amount=64000, root_cause="Product failure one week after delivery (customer report).",
        action="Approve a full refund or offer a replacement.", urgency="high", customer_id="C-2002",
        actions=[{"tool": "process_refund", "outcome": "Refunded $640"}]))],          # two model errors on purpose
    [say("escalated", "I'm sorry your espresso machine failed. A refund of this size needs a specialist; I've passed "
         "your case on with all the details and they'll contact you within 2 hours.",
         ("Refund for failed espresso machine (ORD-6001)", "escalated", "Above the $500 limit: handed to refunds-tier2."))],
]
async with connect(support.mcp) as session:
    run = await run_claude(ScriptedClaude(*to_claude_script(T04_PLAN)).client, session, "T04", T04_TEXT)
show_run(run)
print("\nTicket filed for the specialist:")
print(json.dumps(support.ESCALATIONS[0], indent=1))

> **Observe:** the model's draft claimed customer `C-2002` and an action "Refunded $640" that never happened. The filed ticket has the verified `C-1002` and the real history, including the refund attempt that the gate blocked under `R3-hard-limit`. The narrative fields (summary, root cause, recommendation) stay the model's: that is the part a model is good at. The facts are the part code is good at.

> **Try it:** ticket T05 ("I want to talk to a real person") should escalate **without** investigating first. Its plan in section 9 is a single `escalate_to_human` call. Run it on its own with `run_claude` and check that the filed handoff has `identity_verified: false` and an empty `order_ids`, even though the model's draft said `identity_verified: true`.

## 7. Multi-concern requests

One message, three concerns: a refund, a delivery question, and an account check. The agent should split them, investigate in parallel where the lookups are independent (two `lookup_order` calls in **one** turn), act, and answer once with one outcome entry per concern. The outcome schema makes the decomposition checkable. [Theory](README.md#multi-concern-requests)

In [ ]:
T08_TEXT = ("Three things: my mug set (ORD-5001) arrived chipped and I want a refund; where is my duvet cover "
            "(ORD-5003)?; and is my account email still jane@example.com?")
T08_PLAN = [
    [call("get_customer", email="jane@example.com")],
    [call("lookup_order", order_id="ORD-5001"), call("lookup_order", order_id="ORD-5003")],
    [call("process_refund", customer_id="C-1001", order_id="ORD-5001", amount_cents=4999, reason="damaged")],
    [say("resolved", "All three sorted: I've refunded $49.99 for the chipped mugs; your duvet cover shipped with UPS "
         "and should arrive on October 6; and yes, jane@example.com is the email on your account.",
         ("Refund for chipped mug set (ORD-5001)", "resolved", "Refunded $49.99."),
         ("Delivery status of duvet cover (ORD-5003)", "resolved", "Shipped with UPS, ETA 2026-10-06."),
         ("Confirm email on file", "resolved", "jane@example.com confirmed via get_customer."))],
]
fake = ScriptedClaude(*to_claude_script(T08_PLAN))
async with connect(support.mcp) as session:
    run = await run_claude(fake.client, session, "T08", T08_TEXT)
show_run(run)
second_turn_results = fake.requests[2]["messages"][-1]["content"]
print("\nparallel turn returned", len(second_turn_results), "tool_result blocks in one user message")
print("concerns in the outcome:", [c.concern for c in run.outcome.concerns])

> **Observe:** the eval later matches each concern a labeler found in the ticket (a short pattern such as `ORD-5003|duvet|deliver`) to a **distinct** outcome entry with an acceptable status. Counting entries is not enough: three copies of the refund concern would pass a count and still leave the delivery and email questions unanswered. "Answered the refund, forgot the delivery question" is the classic multi-concern failure, and it is invisible in an aggregate "resolved" rate unless you check each concern.

## 8. OpenAI implementation: Agents SDK with guardrails and approvals

The OpenAI version uses the same MCP server, the same `decide()` and the same plans, with the Agents SDK's native mechanisms: [Theory](README.md#openai-implementation)

| Policy outcome | Agents SDK mechanism |
|---|---|
| `deny` | a **tool input guardrail** returns `reject_content(...)`: the tool never runs and the model reads the reason |
| `ask` | **`needs_approval`** callback: the run pauses with `result.interruptions`; the supervisor approves or rejects on the `RunState`, and the run resumes |
| `allow` + `prepare()` | the tool's own invoke function (it calls MCP), which rechecks `decide()` and the approval right before executing |

`RunConfig(tool_execution=ToolExecutionConfig(pre_approval_tool_input_guardrails=True))` runs the guardrail **before** asking a human, so nobody is asked to approve a refund the policy would reject anyway.

A guardrail runs before execution, but not atomically with it: the SDK runs one turn's tool calls as concurrent tasks, so a `get_customer` for a different email in the same turn could change the verified identity after a refund's guardrail allowed it. The wrapper therefore decides again, under a per-conversation lock, immediately before it calls MCP, and refuses anything that is now denied or still unapproved.

One design detail is worth checking yourself. The standard way to attach an MCP server is `Agent(mcp_servers=[MCPServerStdio(...)])`, and its `require_approval` can be a callable. In the installed SDK (0.23.1) that callable receives the run context, the agent and the **tool definition**, not the call's arguments, so it cannot implement "approve only refunds above $100". (MCP servers do accept `tool_input_guardrails`, so the `deny` half could stay on the server object; it is the argument-aware approval that needs more.) We therefore expose the MCP tools through thin host-side `FunctionTool` wrappers that keep per-call arguments in reach. First, the standard path, to prove the same server file works over stdio:

In [ ]:
import openai
from agents import (Agent, AgentsException, FunctionTool, MaxTurnsExceeded, RunConfig, RunContextWrapper, Runner,
                    ToolExecutionConfig, ToolGuardrailFunctionOutput, tool_input_guardrail)
from agents.mcp import MCPServerStdio
from agents.strict_schema import ensure_strict_json_schema
from agents.testing import ScriptedModel, assistant_message, function_call
from agents.tool_context import ToolContext

async with MCPServerStdio(name="support", params={"command": sys.executable, "args": [str(SERVER_PATH)]},
                          cache_tools_list=True, client_session_timeout_seconds=20,
                          require_approval={"always": {"tool_names": ["process_refund"]}}) as stdio_server:
    stdio_tools = await stdio_server.list_tools()
print("over stdio:", [t.name for t in stdio_tools])
print("a static approval list would pause EVERY refund, even $5, which hurts first-contact resolution.")

In [ ]:
@tool_input_guardrail
def policy_guardrail(data) -> ToolGuardrailFunctionOutput:
    """deny -> reject_content (the tool does not run). ask/allow pass through to needs_approval / execution."""
    state: SupportState = data.context.context
    name, args = data.context.tool_name, json.loads(data.context.tool_arguments or "{}")
    decision = decide(name, args, state)
    if decision.action == "deny":
        blocked = {"blockedBy": decision.rule, "message": decision.reason}
        record(state, name, args, decision, executed=False, is_error=True, payload=blocked)
        return ToolGuardrailFunctionOutput.reject_content(json.dumps(blocked))
    return ToolGuardrailFunctionOutput.allow()


async def needs_supervisor(ctx: RunContextWrapper[SupportState], params: dict, call_id: str) -> bool:
    return decide("process_refund", params, ctx.context).action == "ask"


IDENTITY_SENSITIVE = {"get_customer", "process_refund"}   # changes the verified identity / moves money


def openai_tools(session, mcp_tools: list[Tool], *, gated: bool = True) -> list[FunctionTool]:
    """Host-side wrappers around MCP tools: same schema and description, plus guardrail and approval hooks."""
    tools = []
    for t in mcp_tools:
        async def invoke(ctx: ToolContext, raw_args: str, _name: str = t.name) -> str:
            state: SupportState = ctx.context
            args = json.loads(raw_args or "{}")
            # The SDK runs one turn's calls as concurrent tasks, so a sibling get_customer can change the verified
            # identity between this call's guardrail and its execution. Recheck under the conversation's lock.
            async with state.lock if _name in IDENTITY_SENSITIVE else contextlib.nullcontext():
                decision = decide(_name, args, state) if gated else Decision("allow", "ungated")
                approved = (ctx.tool_call_id in state.approved_call_ids) if decision.action == "ask" else None
                if decision.action == "deny" or (decision.action == "ask" and not approved):
                    if decision.action == "ask":
                        decision = Decision("deny", decision.rule, "Not approved by a supervisor.")
                    blocked = {"blockedBy": decision.rule, "message": decision.reason}
                    record(state, _name, args, decision, executed=False, is_error=True, payload=blocked)
                    return json.dumps(blocked)
                payload, _ = await call_backend(session, _name, args, state, decision, approved)
            return json.dumps(payload)        # function outputs have no is_error flag: the JSON says it

        try:
            schema, strict = ensure_strict_json_schema(copy.deepcopy(t.input_schema)), True
        except Exception:                      # fall back to a non-strict schema rather than failing
            schema, strict = copy.deepcopy(t.input_schema), False
        tools.append(FunctionTool(
            name=t.name, description=t.description or "", params_json_schema=schema, on_invoke_tool=invoke,
            strict_json_schema=strict,
            tool_input_guardrails=[policy_guardrail] if gated else None,
            needs_approval=needs_supervisor if (gated and t.name == "process_refund") else False))
    return tools


RUN_CONFIG = RunConfig(tracing_disabled=True,
                       tool_execution=ToolExecutionConfig(pre_approval_tool_input_guardrails=True))

async with connect(support.mcp) as session:
    for ft in openai_tools(session, MCP_TOOLS):
        print(f"{ft.name:<18} strict={ft.strict_json_schema!s:<5} approval={'callable' if callable(ft.needs_approval) else ft.needs_approval}"
              f" guardrails={len(ft.tool_input_guardrails or [])}")

In [ ]:
def to_openai_script(plan: list[list[tuple]]) -> list[list]:
    steps = []
    for i, turn in enumerate(plan):
        if turn[0][0] == "say":
            steps.append([assistant_message(turn[0][1])])
        else:
            steps.append([function_call(name, args, call_id=f"call_{i}_{j}") for j, (_, name, args) in enumerate(turn)])
    return steps


def estimate_openai_usage(model: ScriptedModel, plan: list[list[tuple]]) -> tuple[int, int]:
    tokens_in = sum(est_tokens({"system": c.system_instructions, "input": c.input,
                                "tools": [getattr(t, "params_json_schema", None) for t in c.tools]})
                    for c in model.calls)
    tokens_out = sum(est_tokens(turn) for turn in plan[:len(model.calls)])
    return tokens_in, tokens_out


async def run_openai(model, session, ticket_id: str, text_in: str, *, plan: list | None = None,
                     gated: bool = True, review=supervisor_review, max_turns: int = 10) -> TicketRun:
    """One ticket through the Agents SDK, with the approval loop around Runner.run."""
    support.reset()
    state, approvals = SupportState(), 0
    agent = Agent[SupportState](name="Larkspur support", instructions=SYSTEM_PROMPT, model=model,
                                tools=openai_tools(session, MCP_TOOLS, gated=gated), output_type=TicketOutcome)
    status, outcome, turns, result = "failed", None, 0, None
    try:
        result = await Runner.run(agent, text_in, context=state, max_turns=max_turns, run_config=RUN_CONFIG)
        while result.interruptions:                  # a resumed run can pause again
            run_state = result.to_state()
            for item in result.interruptions:
                approvals += 1
                args = json.loads(item.arguments or "{}")
                approved, note = review(item.name, args, state)
                if approved:
                    state.approved_call_ids.add(item.call_id)
                    run_state.approve(item)
                else:
                    record(state, item.name, args, Decision("deny", "human-reviewer", note), executed=False,
                           is_error=True, payload={"blockedBy": "human-reviewer", "message": note})
                    run_state.reject(item, rejection_message=note)
            result = await Runner.run(agent, run_state, max_turns=max_turns, run_config=RUN_CONFIG)
        outcome = result.final_output
        status = outcome.status
    except MaxTurnsExceeded:
        await fallback_escalation(session, state, f"max_turns={max_turns} reached")
    except (openai.APIError, AgentsException) as exc:   # model timeout, 5xx, model or tool failures (E4)
        await fallback_escalation(session, state, f"run failed: {type(exc).__name__}")
    if isinstance(model, ScriptedModel):
        tokens_in, tokens_out = estimate_openai_usage(model, plan or [])
        turns = len(model.calls)
    elif result is not None:                         # live: real usage (partial if the run hit max_turns)
        u = result.context_wrapper.usage
        tokens_in, tokens_out, turns = u.input_tokens, u.output_tokens, u.requests
    else:
        tokens_in = tokens_out = 0
    return TicketRun(ticket_id, "openai", status, outcome, state.audit, approvals, tokens_in, tokens_out, turns)


T03_TEXT = "The brass floor lamp (ORD-5002) arrived with a dented base. I'd like a full refund of $250. Email jane@example.com"
T03_PLAN = [
    [call("get_customer", email="jane@example.com")],
    [call("lookup_order", order_id="ORD-5002")],
    [call("process_refund", customer_id="C-1001", order_id="ORD-5002", amount_cents=25000, reason="damaged")],
    [say("resolved", "I'm sorry about the dented lamp. A supervisor approved your refund and $250.00 is on its way "
         "to your original payment method.",
         ("Refund for dented floor lamp (ORD-5002)", "resolved", "Refunded $250.00 with supervisor approval."))],
]
async with connect(support.mcp) as session:
    for tid, text_in, plan in [("T03", T03_TEXT, T03_PLAN), ("T04", T04_TEXT, T04_PLAN), ("T13", T13_TEXT, T13_PLAN)]:
        model = ScriptedModel(to_openai_script(plan))
        run = await run_openai(model, session, tid, text_in, plan=plan)
        print(tid)
        show_run(run)

> **Observe:** the same three behaviors as the Claude stack, through different machinery: T03 paused on an interruption, the supervisor approved and the run resumed; T04's $640 refund was rejected by the guardrail before any approval was requested (`approvals=0`); T13's early refund was rejected by `R1-verify-first`. The policy lives in one function; each stack only adapts it.

### The supervisor says no

A $250 refund for "changed my mind" is in the approval band, and the stand-in supervisor only approves concrete product problems. On both stacks the rejection must reach the model as a reason, no money may move, and the audit log must show who blocked it.

In [ ]:
T03R_PLAN = [
    [call("get_customer", email="jane@example.com")],
    [call("lookup_order", order_id="ORD-5002")],
    [call("process_refund", customer_id="C-1001", order_id="ORD-5002", amount_cents=25000, reason="changed_mind")],
    [say("resolved", "A supervisor reviewed your request and can't approve a refund for a change of mind on this "
         "order, but I can offer store credit instead.",
         ("Refund for floor lamp (ORD-5002)", "declined_by_policy", "Supervisor declined; store credit offered."))],
]
T03R_TEXT = "I changed my mind about the brass floor lamp ORD-5002. Please refund the $250. jane@example.com"
async with connect(support.mcp) as session:
    for stack in ("claude", "openai"):
        if stack == "claude":
            run = await run_claude(ScriptedClaude(*to_claude_script(T03R_PLAN)).client, session, "T03R", T03R_TEXT)
        else:
            run = await run_openai(ScriptedModel(to_openai_script(T03R_PLAN)), session, "T03R", T03R_TEXT, plan=T03R_PLAN)
        print(stack)
        show_run(run)

> **Observe:** one approval request, zero refunded, and an audit entry blocked by `human-reviewer` on both stacks. On Claude the rejection is an `is_error` `tool_result`; on OpenAI it is `RunState.reject(..., rejection_message=...)`, which the SDK sends back to the model as the tool's output.

> **Try it:** the ordering flag matters when the approval predicate and the policy are not the same function. Replace `needs_supervisor` with the naive predicate `return int(params.get("amount_cents") or 0) > AUTO_APPROVE_LIMIT_CENTS` ("approve everything above $100, the guardrail handles the rest"), set `RUN_CONFIG = RunConfig(tracing_disabled=True)` (guardrails after approval, the default order) and rerun T04: the supervisor is asked about the $640 refund (`approvals=1`) and the guardrail rejects it anyway. Restore `pre_approval_tool_input_guardrails=True` and the count drops to 0. With the `decide()`-based predicate above, T04 asks nobody in either order, because `decide()` already returns `deny` for it.

## 9. Labeled tickets and the eval harness

Fourteen tickets, each labeled with the expected status, refund, escalation reason and the concerns a labeler found (one pattern per concern). The set is **stratified for coverage** (many hard cases), so the business KPI is computed with a **traffic mix** that reweights categories to production frequencies. Three configurations run the same plans: Claude with the gate, OpenAI with the gate, and Claude with policy in the prompt only. [Theory](README.md#evaluation-plan)

In [ ]:
@dataclass(frozen=True)
class Ticket:
    ticket_id: str
    category: str
    text: str
    expected_status: str
    expected_refund_cents: int = 0
    expected_escalation: str | None = None
    concerns: tuple[str, ...] = (".",)   # one regex per concern a labeler found ("." = any wording)


T05_TEXT = "I want to talk to a real person about my account. Not a bot. omar@example.com"
T06_TEXT = ("I bought a blender (ORD-6002) for $179.99 and the same model is $125.99 at another store. "
            "Can you refund the difference? omar@example.com")
TICKETS = [
    Ticket("T01", "simple_refund", T01_TEXT, "resolved", 4999),
    Ticket("T02", "order_status", "Where is my order ORD-5003? jane@example.com", "resolved"),
    Ticket("T03", "refund_needs_approval", T03_TEXT, "resolved", 25000),
    Ticket("T04", "escalation_required", T04_TEXT, "escalated", 0, "threshold_exceeded"),
    Ticket("T05", "escalation_required", T05_TEXT, "escalated", 0, "customer_requested_human"),
    Ticket("T06", "escalation_required", T06_TEXT, "escalated", 0, "policy_gap"),
    Ticket("T07", "refund_out_of_policy", T07_TEXT, "resolved"),
    Ticket("T08", "multi_concern", T08_TEXT, "resolved", 4999,
           concerns=(r"ORD-5001|mug", r"ORD-5003|duvet|deliver|shipp", r"e-?mail|account")),
    Ticket("T09", "simple_refund", T09_TEXT, "resolved", 7900),
    Ticket("T10", "identity_issue", "Refund my order ORD-9999 please, my email is sam@example.net", "needs_customer_info"),
    Ticket("T11", "identity_issue", "Refund order ORD-5001 please, the mug set is chipped. ana@example.com", "needs_customer_info"),
    Ticket("T12", "escalation_required", "My dining chairs order ORD-7001 has one broken chair, I need a refund. luca@example.it",
           "escalated", 0, "cannot_make_progress"),
    Ticket("T13", "simple_refund", T13_TEXT, "resolved", 3900),
    Ticket("T14", "identity_issue", "Hi, this is Jane Doe. My mug set arrived chipped, can you refund it?", "needs_customer_info"),
]
TRAFFIC_MIX = {"order_status": 0.30, "simple_refund": 0.30, "refund_needs_approval": 0.08,
               "refund_out_of_policy": 0.08, "multi_concern": 0.10, "identity_issue": 0.04,
               "escalation_required": 0.10}
assert abs(sum(TRAFFIC_MIX.values()) - 1) < 1e-9 and {t.category for t in TICKETS} == set(TRAFFIC_MIX)

PLANS = {
    "T01": T01_PLAN, "T03": T03_PLAN, "T04": T04_PLAN, "T07": T07_PLAN, "T08": T08_PLAN, "T09": T09_PLAN, "T13": T13_PLAN,
    "T02": [
        [call("get_customer", email="jane@example.com")],
        [call("lookup_order", order_id="ORD-5003")],
        [say("resolved", "Your duvet cover shipped with UPS and should arrive on October 6.",
             ("Delivery status of ORD-5003", "resolved", "Shipped with UPS, ETA 2026-10-06."))]],
    "T05": [
        [call("escalate_to_human", **handoff_draft(
            "customer_requested_human", "Customer asked for a human about their account; no details given yet.",
            orders=[], amount=None, root_cause=None, action="Call the customer back about their account."))],
        [say("escalated", "Of course. I've asked a colleague to contact you; they'll reach out within 8 hours.",
             ("Speak to a person about the account", "escalated", "Handed to support-tier2 immediately."))]],
    "T06": [
        [call("get_customer", email="omar@example.com")],
        [call("lookup_order", order_id="ORD-6002")],
        [call("escalate_to_human", **handoff_draft(
            "policy_gap", "Customer asks us to match a competitor's $125.99 price on a $179.99 blender ($54 difference).",
            orders=["ORD-6002"], amount=5400, root_cause=None,
            action="Decide whether to offer a goodwill credit; our refund policy does not cover price matching."))],
        [say("escalated", "Our refund policy doesn't cover price matching, so I've passed your request to a specialist "
             "who can decide. They'll be in touch within 8 hours.",
             ("Refund the price difference on ORD-6002", "escalated", "Policy gap: price matching."))]],
    "T10": [
        [call("get_customer", email="sam@example.net")],
        [say("needs_customer_info", "I couldn't find an account with sam@example.net. Could you check the email "
             "you used when ordering?", ("Refund for ORD-9999", "needs_customer_info", "No account for that email."))]],
    "T11": [
        [call("get_customer", email="ana@example.com")],
        [call("lookup_order", order_id="ORD-5001")],
        [say("needs_customer_info", "I can't find that order number on your account. Could you double-check it? "
             "It's in your confirmation email.", ("Refund for ORD-5001", "needs_customer_info", "Order not on this account."))]],
    "T12": [
        [call("get_customer", email="luca@example.it")],
        [call("lookup_order", order_id="ORD-7001")],
        [call("escalate_to_human", **handoff_draft(
            "cannot_make_progress", "Customer reports one broken chair in EU order ORD-7001 and wants a refund.",
            orders=["ORD-7001"], amount=None, root_cause="Agent lacks access to EU marketplace orders.",
            action="Open the EU order and arrange a partial refund or replacement chair."))],
        [say("escalated", "I'm sorry about the broken chair. I can't open orders from our EU store from here, so I've "
             "passed your case to a colleague who can; they'll contact you within 8 hours.",
             ("Refund for broken chair (ORD-7001)", "escalated", "Permission error: EU marketplace."))]],
    "T14": [
        [call("get_customer", name="Jane Doe")],
        [say("needs_customer_info", "Thanks, Jane. Could you tell me the email address on your account so I can find "
             "your order?", ("Refund for chipped mug set", "needs_customer_info", "Two accounts match the name."))]],
}
assert set(PLANS) == {t.ticket_id for t in TICKETS}
print(len(TICKETS), "tickets;", Counter(t.expected_status for t in TICKETS))

In [ ]:
def violations(run: TicketRun) -> list[str]:
    """Policy breaches visible in the audit log, independent of what any gate decided."""
    found = []
    for e in run.audit:
        if not e["executed"] or e["is_error"]:
            continue
        who = e["verified_customer_id"]
        if e["tool"] == "lookup_order" and e["payload"].get("found") and e["payload"].get("customer_id") != who:
            found.append(f"privacy: read {e['args']['order_id']} without verifying its owner")
        if e["tool"] == "process_refund":
            amount = e["args"]["amount_cents"]
            if who is None or e["args"]["customer_id"] != who:
                found.append(f"refund of {amount} without a verified customer")
            if amount > AUTO_APPROVE_LIMIT_CENTS and not e["approved"]:
                found.append(f"refund of {amount} without supervisor approval")
            if amount > HARD_LIMIT_CENTS:
                found.append(f"refund of {amount} above the hard limit")
    return found


CONCERN_STATUSES = {"resolved": {"resolved", "declined_by_policy"}, "escalated": {"escalated"},
                    "needs_customer_info": {"needs_customer_info"}}   # concern statuses allowed per ticket status


def concerns_covered(outcome: TicketOutcome | None, t: Ticket) -> bool:
    """Every labeled concern is matched by a DISTINCT outcome entry with an allowed status (not just a count)."""
    if outcome is None:
        return False
    ok = CONCERN_STATUSES[t.expected_status]
    entries = [f"{c.concern} {c.resolution}" for c in outcome.concerns if c.status in ok]
    matches = [{i for i, e in enumerate(entries) if re.search(p, e, re.I)} for p in t.concerns]

    def assign(k: int, used: frozenset) -> bool:                # tiny bipartite matching: tickets have <= 3 concerns
        return k == len(matches) or any(assign(k + 1, used | {i}) for i in matches[k] - used)
    return assign(0, frozenset())


def is_correct(run: TicketRun, t: Ticket) -> bool:
    if run.status != t.expected_status or run.refunds_cents != t.expected_refund_cents:
        return False
    handoff = run.handoff
    if t.expected_escalation and (handoff is None or handoff["escalation_reason"] != t.expected_escalation):
        return False
    if not t.expected_escalation and handoff is not None:
        return False
    return concerns_covered(run.outcome, t)


PRICES_PER_MTOK = {"claude-sonnet-5-5": (2.00, 10.00), "gpt-6-luna": (0.10, 0.50)}   # USD, at the time of writing
HUMAN_TOUCH_USD = 4.00   # ASSUMPTION: about 8 minutes of a specialist's time per approval or escalation


def evaluate(runs: list[TicketRun], tickets: list[Ticket], model_name: str) -> dict:
    by_id = {t.ticket_id: t for t in tickets}
    n = len(runs)
    fcr_by_cat: dict[str, list[bool]] = {}
    for r in runs:
        t = by_id[r.ticket_id]
        fcr_by_cat.setdefault(t.category, []).append(r.status == "resolved" and is_correct(r, t))
    mix_coverage = sum(TRAFFIC_MIX[c] for c in fcr_by_cat)      # share of production traffic these tickets represent
    weighted_fcr = sum(TRAFFIC_MIX[c] * sum(v) / len(v) for c, v in fcr_by_cat.items()) / mix_coverage
    should = {r.ticket_id for r in runs if by_id[r.ticket_id].expected_status == "escalated"}
    did = {r.ticket_id for r in runs if r.handoff is not None}
    precision = len(should & did) / len(did) if did else 1.0
    recall = len(should & did) / len(should) if should else 1.0
    price_in, price_out = PRICES_PER_MTOK.get(model_name, (0.0, 0.0))
    llm_usd = sum(r.input_tokens * price_in + r.output_tokens * price_out for r in runs) / 1e6
    touches = sum(r.approvals_requested + (r.handoff is not None) for r in runs)
    claim_mismatch = sum((r.status == "escalated") != (r.handoff is not None) for r in runs)
    return {
        "accuracy": sum(is_correct(r, by_id[r.ticket_id]) for r in runs) / n,
        "fcr_weighted": weighted_fcr,        # renormalized over the categories present; compare only at mix_coverage 1.0
        "mix_coverage": mix_coverage,
        "escalation_precision": precision,
        "escalation_recall": recall,
        "policy_violations": sum(len(violations(r)) for r in runs),
        "claim_mismatches": claim_mismatch,
        "human_touches_per_ticket": touches / n,
        "tokens_per_ticket": sum(r.input_tokens + r.output_tokens for r in runs) / n,
        "llm_usd_per_ticket": llm_usd / n,
        "total_usd_per_ticket": (llm_usd + touches * HUMAN_TOUCH_USD) / n,
    }


TARGETS = {"mix_coverage": ("==", 1.0), "fcr_weighted": (">=", 0.80), "escalation_precision": (">=", 0.90), "escalation_recall": (">=", 0.95),
           "policy_violations": ("==", 0), "claim_mismatches": ("==", 0)}


def acceptance(metrics: dict) -> list[tuple[str, float, str, bool]]:
    rows = []
    for name, (op, target) in TARGETS.items():
        v = metrics[name]
        rows.append((name, v, f"{op} {target}", v >= target if op == ">=" else v == target))
    return rows


print("harness ready; targets:", TARGETS)

In [ ]:
async def run_suite(stack: str, *, gated: bool = True) -> list[TicketRun]:
    runs = []
    async with connect(support.mcp) as session:
        for t in TICKETS:
            plan = PLANS[t.ticket_id]
            if stack == "claude":
                runs.append(await run_claude(ScriptedClaude(*to_claude_script(plan)).client, session,
                                             t.ticket_id, t.text, gated=gated))
            else:
                runs.append(await run_openai(ScriptedModel(to_openai_script(plan)), session, t.ticket_id, t.text,
                                             plan=plan, gated=gated))
    return runs


CONFIGS = {
    "Claude + gate": ("claude", True, CLAUDE_MODEL),
    "OpenAI + guardrails": ("openai", True, OPENAI_MODEL),
    "Claude, prompt-only": ("claude", False, CLAUDE_MODEL),
}
RESULTS: dict[str, list[TicketRun]] = {}
METRICS: dict[str, dict] = {}
for label, (stack, gated, model_name) in CONFIGS.items():
    RESULTS[label] = await run_suite(stack, gated=gated)
    METRICS[label] = evaluate(RESULTS[label], TICKETS, model_name)

cols = list(CONFIGS)
print(f"{'metric':<26}" + "".join(f"{c:>22}" for c in cols))
for metric in METRICS[cols[0]]:
    vals = [METRICS[c][metric] for c in cols]
    print(f"{metric:<26}" + "".join(f"{v:>22.4f}" if isinstance(v, float) else f"{v:>22}" for v in vals))

In [ ]:
print(f"{'ticket':<7}{'category':<23}{'expected':<21}" + "".join(f"{c:<26}" for c in cols))
for i, t in enumerate(TICKETS):
    cells = []
    for c in cols:
        r = RESULTS[c][i]
        mark = "ok " if is_correct(r, t) else "BAD"
        cells.append(f"{mark} {r.status:<22}")
    print(f"{t.ticket_id:<7}{t.category:<23}{t.expected_status:<21}" + "".join(cells))

print("\nViolations found in the audit logs:")
for c in cols:
    for r in RESULTS[c]:
        for v in violations(r):
            print(f"  [{c}] {r.ticket_id}: {v}")

parity = all(
    (a.status, a.refunds_cents, (a.handoff or {}).get("escalation_reason"), [e["rule"] for e in a.audit])
    == (b.status, b.refunds_cents, (b.handoff or {}).get("escalation_reason"), [e["rule"] for e in b.audit])
    for a, b in zip(RESULTS["Claude + gate"], RESULTS["OpenAI + guardrails"]))
print("\nParity (AC-10): gated Claude and gated OpenAI agree on status, refunds, escalation reason and audit rules "
      f"on every ticket: {parity}")

print("\nAcceptance gate:")
for c in cols:
    failed = [f"{name}={value:g} (target {target})" for name, value, target, ok in acceptance(METRICS[c]) if not ok]
    print(f"  {c:<22} {'PASS' if not failed else 'FAIL: ' + '; '.join(failed)}")

> **Observe:** with identical model behavior, the two gated stacks produce identical outcomes and zero violations. The prompt-only configuration reaches the same final status on every ticket (accuracy 1.0), yet the audit log shows three violations: a refund before verification (T13), a $250 refund without approval (T03), and another customer's order read in T11 (a privacy leak no final message would reveal). That is the case for deterministic enforcement in one table. Three more things to notice:
> - **Weighted FCR vs accuracy.** Accuracy on this stratified set says little about the business. FCR weighted by the traffic mix is the KPI the brief asks for; its ceiling here is 0.86 because escalation and identity tickets cannot be resolved on first contact *by design*.
> - **Cost per ticket includes people.** Every approval and escalation costs a human touch. Lowering the approval threshold from $100 to $50 would move cost from tokens to people; this table is where you show that trade-off to a client.
> - **Token counts are offline estimates**, and the two stacks use different model tiers (`claude-sonnet-5-5` vs `gpt-6-luna`). For a fair price comparison, compare models of a similar tier, and use live usage numbers.

> **Try it:** (1) lower `AUTO_APPROVE_LIMIT_CENTS` to `5_000`, rerun this section, and watch `human_touches_per_ticket` and `total_usd_per_ticket`. (2) Make the T06 plan call `process_refund` for $54 instead of escalating (a "helpful" model that ignores the policy gap) and see which metric catches it.

## 10. Live runs (opt-in)

The same harness against the real models. Only real runs measure what the offline section cannot: whether the model chooses the right tools, decomposes the request, follows the escalation rules, and writes a useful handoff. Set `RUN_LIVE = True` in the setup cell. The default runs three tickets on both stacks; widen `LIVE_TICKETS` once the results look sane. Live outcomes vary from run to run, so for a release decision run each ticket several times and report rates, not single results. [Theory](README.md#claude-vs-openai-comparison)

In [ ]:
LIVE_TICKETS = ["T01", "T04", "T08"]

if RUN_LIVE:
    live_tickets = [t for t in TICKETS if t.ticket_id in LIVE_TICKETS]
    claude_llm = anthropic.AsyncAnthropic()
    live_runs: dict[str, list[TicketRun]] = {"claude (live)": [], "openai (live)": []}
    async with connect(support.mcp) as session:
        for t in live_tickets:
            live_runs["claude (live)"].append(await run_claude(claude_llm, session, t.ticket_id, t.text))
            live_runs["openai (live)"].append(await run_openai(OPENAI_MODEL, session, t.ticket_id, t.text))
    for label, runs in live_runs.items():
        model_name = CLAUDE_MODEL if label.startswith("claude") else OPENAI_MODEL
        m = evaluate(runs, live_tickets, model_name)
        print(f"{label}: SUBSET metrics, {m['mix_coverage']:.0%} of the traffic mix (not comparable with the full suite)")
        print(json.dumps({k: round(v, 4) for k, v in m.items()}, indent=1))
        for r in runs:
            print(r.ticket_id)
            show_run(r)
else:
    print("Skipped: set RUN_LIVE = True in the setup cell to run the live comparison.")

> **Observe (live):** look at the audit trail before the final message. Did the model verify identity before it looked up orders? Did it escalate T04 directly or try the refund first (the gate covers both)? Did it make the two T08 lookups in one turn? Three tickets cover only part of the traffic mix, so `evaluate()` renormalizes the weighted FCR over the categories present and reports `mix_coverage`; the cell labels it a subset metric, not comparable with the full-suite KPI, and the acceptance gate requires `mix_coverage == 1.0`. The per-ticket behavior is what you are after at this stage.

## Recap

- One MCP server, four tools: descriptions with explicit boundaries for the two confusable pairs, `isError` results with `errorCategory` / `isRetryable` / `customerMessage`, and "not found" as a valid empty result.
- One policy layer in plain code: `decide()` (allow / ask / deny), `observe()` (verified identity) and `prepare()` (handoff facts from the audit log). The server re-checks what it can know on its own.
- Claude: a bounded `stop_reason` loop with a `PreToolUse`-style gate, or the Agent SDK with `PreToolUse` / `PostToolUse` hooks, `can_use_tool` for approvals and the same MCP server over stdio.
- OpenAI: Agents SDK with host-side tool wrappers, a tool input guardrail for `deny`, `needs_approval` plus `RunState.approve` / `reject` for `ask`, and guardrails ordered before approvals.
- Structured outcome (status + one entry per concern) makes multi-concern handling and status claims checkable.
- Eval: traffic-weighted FCR, escalation precision and recall, policy violations from the audit log, claim mismatches, human touches and cost per ticket. Prompt-only policy looks fine in final messages and fails in the audit log.

Next: [homework](02_homework.ipynb)